# 07 Judges (LLM-as-judge)

**What this does.** A judge grades target-comp pairs 0/1/2 for evaluation only: it never chooses comps or changes a
list. It is kept independent of the selector by its inputs (website text with citations, no facets, no selector
decision) and by its model. **Decision (Michael, 2026-11-18):** judges stay in the GPT family, so the model line is
clear (Claude selects, GPT judges); a GPT model of similar strength to Sonnet may replace Sol.

**Current judge result:** GPT-6.1 Sol, label-blind, source-citing, with web search, on Barbara's 293 pairs (11-10).

Valuation of the judge filters is in `08_valuation/gpt61sol_293_selection/`; the 10-28 Sol cited grades of the
holdout targets are in `10_final_holdout`.

**Changelog**
- 2026-11-18: created from `v1/PEER_EVALUATION.md`, `peer_evaluation.py` (Kethan's version with explicit pairs),
  `judge_gold_eval.py` and `integration_judge_eval.py` (now `judge_293_eval.py`). The 293-pair evaluation
  reproduces exactly from the judge's saved answers.

In [1]:
import sys
from pathlib import Path
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "v1a" / "current.toml").exists())
sys.path.insert(0, str(ROOT / "v1a" / "common"))
import paths, nbtools as nb
paths.setup()
import pandas as pd
pd.set_option("display.width", 160); pd.set_option("display.max_colwidth", 90); pd.set_option("display.max_rows", 80)
cur = paths.current()
checks = nb.Checks()

In [2]:
import json, cohorts, judge_293_eval as je
E = paths.out("07_judges") / "gpt61sol_293" / "eval_293"
nb.inputs([str((E / f).relative_to(ROOT).as_posix()) for f in ["all_pairs.csv", "summary.json", "selection_metrics.csv"]])

,input,rows,sha256,status
0,outputs-local/v1a/07_judges/gpt61sol_293/eval_293/all_pairs.csv,293.0,f756b87d0db2,matches seed
1,outputs-local/v1a/07_judges/gpt61sol_293/eval_293/summary.json,NaN,ed73b1cc54e8,matches seed
2,outputs-local/v1a/07_judges/gpt61sol_293/eval_293/selection_metrics.csv,3.0,bd92d437848a,matches seed


## 3. Results

### 3.1 GPT-6.1 Sol on Barbara's 293 pairs

In [3]:
summary = json.loads((E / "summary.json").read_text())
pd.Series({k: summary[k] for k in ["pairs_scored", "exact_agreement", "within_one_grade", "peer_binary_agreement",
                                   "mean_signed_grade_error", "web_searches", "saved_agent_exact_on_same_pairs",
                                   "new_agent_exact_on_same_pairs"]}).round(3).to_frame("value")

,value
pairs_scored,293.000
exact_agreement,0.785
within_one_grade,1.000
peer_binary_agreement,0.843
mean_signed_grade_error,0.024
web_searches,77.000
saved_agent_exact_on_same_pairs,0.839
new_agent_exact_on_same_pairs,0.784


In [4]:
conf = pd.read_csv(E / "confusion.csv", index_col=0)
conf.index.name, conf.columns.name = "Barbara", "GPT Sol"
conf

GPT Sol,0,1,2
Barbara,,,
0,61,30,0
1,16,157,5
2,0,12,12


### 3.2 The judge as a filter, on the same pairs (label counts only)

`selection_metrics.csv`: how many of Barbara's direct, adjacent and non-peer pairs a judge-grade filter keeps.

In [5]:
pd.read_csv(E / "selection_metrics.csv").round(3)

,rule,selected_pairs,targets_with_pairs,reviewed_direct_kept,reviewed_adjacent_kept,reviewed_nonpeer_kept,good_recall,selected_good_fraction
0,Reviewed B3 pairs,293,45,24,178,91,1.000,0.689
1,Judge score >= 1,216,44,24,162,30,0.921,0.861
2,Judge score = 2,17,10,12,5,0,0.084,1.000


### 3.3 Round 1: the 0-2 exact judge on the 36 golden-set pairs (10-28, v1 record)

In [6]:
G = paths.V1_OUTPUTS / "golden_set" / "judge_comparison_0_2_exact" / "tables"
pd.read_csv(G / "agreement.csv").round(3)

,reference,labeled_pairs,graded_pairs,matching_pairs,exact_agreement,mean_absolute_grade_error
0,riya,36,36,15,0.417,0.639
1,barbara,36,36,32,0.889,0.111
2,matthew,36,36,26,0.722,0.278
3,majority,32,32,27,0.844,0.156


## 4. Checks

In [7]:
import contextlib, io, tempfile
with tempfile.TemporaryDirectory() as tmp:
    tmp = Path(tmp)
    v = je.verdicts()
    v.to_csv(tmp / "verdicts.csv", index=False)
    with contextlib.redirect_stdout(io.StringIO()):
        s = je.evaluate(tmp)
    checks.add("verdicts rebuild from the 293 saved answers", len(v) == 293 and v["status"].eq("ok").all())
    checks.add("293-pair evaluation reproduces v1 (summary)", s == summary, f"exact {s['exact_agreement']:.3f}")
    checks.add("confusion, per-target and disagreement tables reproduce",
               all(pd.read_csv(tmp / "eval_293" / f, dtype=str).fillna("").equals(pd.read_csv(E / f, dtype=str).fillna(""))
                   for f in ["confusion.csv", "by_target.csv", "disagreements.csv"]))
checks.add("judges cover dev_293 exactly", set(zip(v["target"], v["comp"])) ==
           set(zip(cohorts.load("dev_293")["target_id"], cohorts.load("dev_293")["comp_id"])))
leaks = [p.name for p in paths.out("07_judges").rglob("*") if p.is_file() and p.suffix in (".csv", ".json")
         and any(w in p.read_text(encoding="utf-8", errors="ignore")[:3000] for w in ("ev_to_ebitda", "abs % error", "median_ape"))]
checks.add("no valuation results in 07", not leaks, ", ".join(leaks))
checks.show()

5/5 checks pass


,check,result,detail
0,verdicts rebuild from the 293 saved answers,pass,
1,293-pair evaluation reproduces v1 (summary),pass,exact 0.785
2,"confusion, per-target and disagreement tables reproduce",pass,
3,judges cover dev_293 exactly,pass,
4,no valuation results in 07,pass,


## 5. History (v1)

- **10-27 (v0):** the evidence-only 0-3 judge (`peer_evaluation.py`) on 663 pairs of the size-stratified baseline.
- **10-28:** the 0-2 exact judge on the 36 round-1 pairs (`judge_gold_eval.py`): 89% exact with Barbara, 84% with
  the majority. Sol's cited 0-2 grades of the holdout targets' pairs (`graded_peer_selection.py`) are in
  `10_final_holdout`.
- **11-10 (integration study):** the label-blind GPT-6.1 Sol judge on the 293 pairs (above).
- **11-12 (Kethan, his machine):** GPT-6 Luna 0-3 judge on all 2,438 pairs of his Sol-filled B3 list; `peer_evaluation.py`
  gained `prepare_pairs` for explicit pair lists (ported here).